In [50]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
import matplotlib.ticker as mtick

In [51]:
# load variable datasets
vort = xr.open_mfdataset("datasets/GPI/GPI_EN_calc/post_landmask/*.nc", combine="by_coords")

vmax = xr.open_dataset("datasets/potential_intensity/pi_output_daily_landmasked_v2.nc")

rhum = xr.open_mfdataset("datasets/RHUM/post_processing/post_landmask/yearly/*.nc", combine="by_coords")

shear = xr.open_mfdataset("datasets/u-wind/post_processing/post_landmask/shear_annual/*.nc", combine="by_coords")

/var/folders/d8/dpf8hpcx6l34ky46wf5b_v2h0000gn/T/ipykernel_7710/1524110260.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  vort = xr.open_mfdataset("datasets/GPI/GPI_EN_calc/post_landmask/*.nc", combine="by_coords")
/var/folders/d8/dpf8hpcx6l34ky46wf5b_v2h0000gn/T/ipykernel_7710/1524110260.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'pressure_level' ('pressure_level',) The recommendation is to set join explicitly for this c

In [ ]:
# select variables
vort = vort["abs_vort850"]
vmax = vmax["vmax"]
rhum = rhum["r"]
shear = shear["wind_shear"]


<xarray.DataArray 'r' (time: 13158, pressure_level: 1, lat: 361, lon: 581)> Size: 11GB
dask.array<concatenate, shape=(13158, 1, 361, 581), dtype=float32, chunksize=(153, 1, 361, 581), chunktype=numpy.ndarray>
Coordinates:
  * time            (time) datetime64[ns] 105kB 1940-06-01 ... 2025-10-31
  * pressure_level  (pressure_level) float64 8B 600.0
  * lat             (lat) float64 3kB 90.0 89.75 89.5 89.25 ... 0.5 0.25 0.0
  * lon             (lon) float64 5kB -120.0 -119.8 -119.5 ... 24.5 24.75 25.0
    number          int64 8B 0
Attributes: (12/31)
    GRIB_paramId:                             157
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      209741
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_shortName:                           r
    GRIB_totalNumber:          

In [53]:
# remove the 850hPa dimension from vort & 600hPa from rhum since the other datasets do not have a pressure dimension
vort = vort.squeeze("pressure_level", drop=True)
rhum = rhum.squeeze("pressure_level", drop=True)

# filter date ranges so they all match
vort, vmax, rhum, shear = xr.align(
    vort,
    vmax,
    rhum,
    shear,
    join="inner"
)

# check dimensions
print("vort:", vort.dims, vort.shape)
print("vmax:", vmax.dims, vmax.shape)
print("rhum:", rhum.dims, rhum.shape)
print("shear:", shear.dims, shear.shape)


vort: ('time', 'lat', 'lon') (6763, 37, 57)
vmax: ('time', 'lat', 'lon') (6763, 37, 57)
rhum: ('time', 'lat', 'lon') (6763, 37, 57)
shear: ('time', 'lat', 'lon') (6763, 37, 57)


In [ ]:
# # check
# print("vort:")
# print("NaNs:", vort.isnull().sum().compute().item(), "/", vort.size)

# print("\nvmax:")
# print("NaNs:", vmax.isnull().sum().compute().item(), "/", vmax.size)

# print("\nrhum:")
# print("NaNs:", rhum.isnull().sum().compute().item(), "/", rhum.size)

# print("\nshear:")
# print("NaNs:", shear.isnull().sum().compute().item(), "/", shear.size)

vort:
NaNs: 0 / 0

vmax:
NaNs: 9285268 / 14263167

rhum:
NaNs: 0 / 0

shear:
NaNs: 6424850 / 14263167


In [54]:
# calc GPI using Emanual and Nolan model
# first term
a = (np.abs((10**5)*vort)) ** (3/2)

# second term
b = (rhum/50)**3

# third term
c = (vmax/70)**3

# fourth term
d = (1 + (0.1*shear))**(-2)

# calc GPI
gpi = a * b * c * d


In [55]:
# change attributes before saving
gpi.attrs["long_name"] = "Genesis Potential Index"
gpi.attrs["standard_name"] = "genesis_potential_index"
gpi.attrs["units"] = "1"
gpi.attrs["description"] = (
    "Genesis Potential Index from Emanuel and Nolan (2004) model"
)
gpi.attrs["references"] = "Emanuel and Nolan (2004)"
gpi.attrs["cell_methods"] = "time: monthly means"
gpi.name = 'gpi'

# remove inherited attributes that no longer apply
for attr in [
    "GRIB_id",
    "GRIB_name",
    "level_desc",
    "dataset",
    "parent_stat",
    "statistic",
    "actual_range",
    "valid_range",
]:
    gpi.attrs.pop(attr, None)

print(gpi)


<xarray.DataArray 'gpi' (time: 6763, lat: 37, lon: 57)> Size: 114MB
dask.array<mul, shape=(6763, 37, 57), dtype=float64, chunksize=(153, 37, 57), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 54kB 1981-09-01 1981-09-02 ... 2025-10-01
  * lat          (lat) float64 296B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float64 456B -120.0 -117.5 -115.0 ... 15.0 17.5 20.0
    number       int64 8B 0
    expver       (time) <U4 108kB '0001' '0001' '0001' ... '0001' '0001' '0001'
    spatial_ref  int64 8B 0
Attributes: (12/35)
    stored_direction:                         decreasing
    GRIB_paramId:                             157
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      209741
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    ...                                       ...
    var_desc:                                 Sea Surfa

In [36]:
print("GPI:", gpi.dims, gpi.shape)
print("Vorticity:", vort.dims, vort.shape)
print("Vmax:", vmax.dims, vmax.shape)

GPI: ('time', 'pressure_level', 'lat', 'lon') (6763, 0, 37, 57)
Vorticity: ('time', 'pressure_level', 'lat', 'lon') (6763, 0, 37, 57)
Vmax: ('time', 'lat', 'lon') (6763, 37, 57)


In [56]:
# save to net cdf
gpi.to_netcdf("datasets/GPI/GPI_EN_calc/post_landmask/GPI_EN_output_daily_landmasked.nc")